In [ ]:
import csv, copy, json, math, tarfile, warnings
import numpy as np
import pandas as pd
from itertools import combinations as _combinations
from pathlib import Path
import kaggle_environments as ke

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DAYS = [
    "2026-05-01",
    "2026-05-02",
    "2026-05-03",
    "2026-05-04",
]
MAX_EPISODES_PER_DAY = 50   # top-N by sum_score per day
NB_FUTURE_STEP       = 10   # future planet snapshots per simulation row
NB_PLANETS           = 44   # 40 non-comets + 4 comets
FEATURE_VERSION      = 1    # 1 → 2200 features (44 × 10 × 5)
MAX_SPEED            = 6.0
PLANET_MARGIN        = 0.1
OUTPUT_DIR           = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Step 01: Winner extraction ────────────────────────────────────────────────
def get_winner_data(raw):
    winner_i = int(np.argmax(raw["rewards"]))
    new_data = []
    for step_list in raw["steps"]:
        obs = dict(step_list[winner_i]["observation"])
        obs["action"] = step_list[winner_i]["action"]
        new_data.append(obs)
    return new_data


# ── Step 02: Augment with destination planet ──────────────────────────────────
def _fleet_speed(nb_ships):
    if nb_ships <= 1:
        return 1.0
    ratio = math.log(nb_ships) / math.log(1000.0)
    return 1.0 + (MAX_SPEED - 1.0) * max(0.0, min(1.0, ratio)) ** 1.5


def augment_data(data):
    fleet_rows, planet_rows = [], []
    for obs in data:                          # fixed: was `winner_data` (global bug)
        s = obs["step"]
        for f in obs["fleets"]:
            fleet_rows.append({"step": s, "id": f[0], "owner": f[1],
                                "x": f[2], "y": f[3], "angle": f[4],
                                "from_planet_id": f[5], "ships": f[6]})
        for p in obs["planets"]:
            planet_rows.append({"step": s, "id": p[0], "owner": p[1],
                                 "x": p[2], "y": p[3], "radius": p[4],
                                 "ships": p[5], "production": p[6]})

    if not fleet_rows or not planet_rows:
        return data

    df_fleet   = pd.DataFrame(fleet_rows).drop_duplicates(["step", "id"]).reset_index(drop=True)
    df_planets = pd.DataFrame(planet_rows).drop_duplicates(["step", "id"]).reset_index(drop=True)

    df_fleet_last = (
        df_fleet
        .assign(first_step=lambda df: df.groupby("id")["step"].transform("min"))
        .groupby("id").last().reset_index()
        .assign(
            speed=lambda df: df["ships"].apply(_fleet_speed),
            dx=lambda df: df["speed"] * np.cos(df["angle"]),
            dy=lambda df: df["speed"] * np.sin(df["angle"]),
            next_step=lambda df: df["step"] + 1,
        )
        .merge(df_planets, left_on="next_step", right_on="step",
               suffixes=("_fleet", "_planet"), how="inner")
        .assign(
            t=lambda df: (
                (df["dx"] * (df["x_planet"] - df["x_fleet"]) +
                 df["dy"] * (df["y_planet"] - df["y_fleet"])) / df["speed"] ** 2
            ).clip(lower=0, upper=1),
            distance=lambda df: np.sqrt(
                (df["x_fleet"] + df["dx"] * df["t"] - df["x_planet"]) ** 2 +
                (df["y_fleet"] + df["dy"] * df["t"] - df["y_planet"]) ** 2),
            on_target=lambda df: df["distance"] < df["radius"] + PLANET_MARGIN,
            angle_3=lambda df: df["angle"].round(3),
        )
        .query("on_target")
        .groupby("id_fleet").first()
        .set_index(["first_step", "from_planet_id", "angle_3", "ships_fleet"])
    )
    fleet_arrival = df_fleet_last["id_planet"].to_dict()

    for step_i, obs in enumerate(data):
        if step_i == 0:
            continue
        new_actions = []
        for from_pid, angle, ships in obs["action"]:
            key = (step_i, from_pid, round(angle, 3), ships)
            if key in fleet_arrival:
                new_actions.append([from_pid, angle, ships, fleet_arrival[key]])
        data[step_i - 1]["action"] = new_actions
    return data


# ── Step 03: Combinatorial augmentation ──────────────────────────────────────
def generate_combinations(obs):
    actions = obs["action"]
    nb = len(actions)
    rows = []
    for nb_done in range(nb):
        for done_idx in _combinations(range(nb), nb_done):
            done = [actions[i] for i in done_idx]
            remaining = [actions[i] for i in range(nb) if i not in done_idx]
            for action_to_do in remaining:
                rows.append({"obs": obs, "done_set": done, "action_to_do": action_to_do})
    rows.append({"obs": obs, "done_set": actions, "action_to_do": None})
    return rows


# ── Step 04: Env simulation ───────────────────────────────────────────────────
def simulate_futures(obs, done_set, nb_future_step=NB_FUTURE_STEP):
    env = ke.make("orbit_wars", debug=False)
    env.reset()
    for key, val in obs.items():
        env.state[0].observation[key] = val
        env.state[1].observation[key] = val
    game_actions = [a[:3] for a in done_set]   # strip dest_planet_id if present
    env.step([game_actions, []])
    snapshots = []
    last = copy.deepcopy(list(env.state[0].observation.get("planets", [])))
    for _ in range(nb_future_step):
        if env.done:
            snapshots.append(last)
            continue
        env.step([[], []])
        last = copy.deepcopy(list(env.state[0].observation.get("planets", [])))
        snapshots.append(last)
    return snapshots


# ── Feature extraction ────────────────────────────────────────────────────────
def build_slot_map(obs):
    comet_ids = sorted(obs.get("comet_planet_ids", []))
    comet_set = set(comet_ids)
    all_ids = [p[0] for p in obs["planets"]]
    non_comet_ids = sorted(pid for pid in all_ids if pid not in comet_set)
    return non_comet_ids, comet_ids


def planet_id_to_slot(planet_id, non_comet_ids, comet_ids):
    if planet_id in comet_ids:
        return 40 + comet_ids.index(planet_id)
    if planet_id in non_comet_ids:
        return non_comet_ids.index(planet_id)
    return -1


def extract_features(future_planets, non_comet_ids, comet_ids, version=1):
    feat = np.zeros((NB_FUTURE_STEP, NB_PLANETS, 5), dtype=np.float32)
    for step_i, planets in enumerate(future_planets):
        for p in planets:
            pid, owner, x, y, radius, ships, prod = p[0], p[1], p[2], p[3], p[4], p[5], p[6]
            slot = planet_id_to_slot(pid, non_comet_ids, comet_ids)
            if slot < 0:
                continue
            feat[step_i, slot] = [owner, ships, x, y, prod]
    return feat.flatten()   # shape: (2200,) for version=1

In [ ]:
all_X, all_yf, all_yt, all_ys = [], [], [], []
episode_count = 0

for day in DAYS:
    slug = f"orbit-wars-top10-episodes-{day}"
    root = Path(f"/kaggle/input/{slug}")
    if not root.exists():
        print(f"[SKIP] {root} not found — add it via + Add Data on Kaggle")
        continue

    with open(root / "manifest.csv") as f:
        manifest = list(csv.DictReader(f))

    to_process = manifest[:MAX_EPISODES_PER_DAY]
    print(f"\nDay {day}: processing {len(to_process)} / {len(manifest)} episodes")

    with tarfile.open(root / "episodes.tar.gz", "r:gz") as tar:
        for row_idx, row in enumerate(to_process):
            ep_id = row["episode_id"]
            try:
                member = tar.getmember(f"episodes/{ep_id}.json")
                raw    = json.load(tar.extractfile(member))

                # Steps 01 → 02
                winner_data = get_winner_data(raw)
                winner_data = augment_data(winner_data)

                # Step 03: combinations (skip obs with no actions)
                rows = []
                for obs in winner_data:
                    if obs.get("action"):
                        rows.extend(copy.deepcopy(generate_combinations(obs)))

                if not rows:
                    continue

                # Step 04: simulate futures
                for r in rows:
                    r["future_planets"] = simulate_futures(
                        r["obs"], r["done_set"], NB_FUTURE_STEP
                    )

                # Feature extraction — per-episode slot map
                non_comet_ids, comet_ids = build_slot_map(rows[0]["obs"])
                for r in rows:
                    feat   = extract_features(r["future_planets"], non_comet_ids, comet_ids)
                    action = r["action_to_do"]
                    if action is None:
                        all_X.append(feat); all_yf.append(44.0)
                        all_yt.append(np.nan); all_ys.append(np.nan)
                    elif len(action) == 4:
                        from_pid, _angle, ships, to_pid = action
                        fs = planet_id_to_slot(from_pid, non_comet_ids, comet_ids)
                        ts = planet_id_to_slot(to_pid,   non_comet_ids, comet_ids)
                        all_X.append(feat); all_yf.append(float(fs))
                        all_yt.append(float(ts)); all_ys.append(float(ships))

                episode_count += 1
                if (row_idx + 1) % 10 == 0:
                    print(f"  {row_idx+1}/{len(to_process)} eps | rows so far: {len(all_X):,}")

            except Exception as e:
                print(f"  ERROR ep {ep_id}: {e}")
                continue

print(f"\nDone. {episode_count} episodes processed, {len(all_X):,} total rows.")

In [ ]:
X       = np.array(all_X,  dtype=np.float32)
y_from  = np.array(all_yf, dtype=np.float32)
y_to    = np.array(all_yt, dtype=np.float32)
y_ships = np.array(all_ys, dtype=np.float32)

np.save(OUTPUT_DIR / "X.npy",       X)
np.save(OUTPUT_DIR / "y_from.npy",  y_from)
np.save(OUTPUT_DIR / "y_to.npy",    y_to)
np.save(OUTPUT_DIR / "y_ships.npy", y_ships)

print(f"Saved to {OUTPUT_DIR}")
print(f"  X shape      : {X.shape}")
print(f"  y_from shape : {y_from.shape}")

In [ ]:
stop_mask    = y_from == 44
invalid_mask = np.isnan(y_to) & ~stop_mask   # non-stop rows with missing dest

print(f"Total rows      : {len(X):,}")
print(f"Stop rows       : {stop_mask.sum():,}  ({100*stop_mask.mean():.1f}%)")
print(f"Invalid rows    : {invalid_mask.sum():,}")
print(f"Action rows     : {(~stop_mask & ~invalid_mask).sum():,}")
print(f"\ny_from distribution (slot → count):")
for slot in sorted(np.unique(y_from[~np.isnan(y_from)]).astype(int)):
    count = (y_from == slot).sum()
    label = "stop" if slot == 44 else str(slot)
    print(f"  slot {label:>3}: {count:,}")